# GNSS-IR processing for the manuscript

This notebook retrieves reflector heights, corrects the GNSS surface-elevation records, and summarizes retrieval counts.

In [ ]:
# -*- coding: utf-8 -*-
"""
Multi-station GNSS-IR SNR-to-RH processor with transparent quality classification.

This version uses ThreadPoolExecutor rather than ProcessPoolExecutor so it can
run reliably from Jupyter on macOS without __main__ pickling errors.

Stations processed
------------------
GZ01, GZ02, GZ06, GZ09, GZ13, GZ17, GZ18
LA01-LA18
MG04-MG07

Processing years
----------------
YEAR_MIN through YEAR_MAX below.

Outputs
-------
A separate folder is created for each station under OUT_ROOT.

For each station:
    <station>_arc_retrievals_all.csv
    <station>_arc_retrievals_valid.csv
    <station>_arc_retrievals_rejected.csv
    <station>_daily_summary_classified.csv
    <station>_filtered_comparison.png

Combined outputs at OUT_ROOT:
    all_stations_daily_summary_classified.csv
    all_stations_processing_summary.csv

Filtering
---------
The original power threshold is retained, then the following additional
quality-control rules are applied:

1. Separate continuous rising and setting arcs.
2. Reject search-boundary peaks.
3. Require primary/secondary peak ratio >= 2.
4. Require primary/background ratio >= 20.
5. Require retrieved RH >= 0.45 m.
6. Require at least 1.5 estimated cycles.
7. Remove preliminary-valid arcs more than 0.30 m from the daily median.
8. Require at least 5 final arcs per day.
9. Remove the final daily RH when fewer than 2filtered of power-accepted arcs remain.

Raw and rejected retrievals are preserved. Only RH_surface_height_m is set
to NaN when daily quality requirements fail.
"""

from __future__ import annotations

import datetime as dt
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, lombscargle

# =====================================================================
# USER SETTINGS
# =====================================================================

STATIONS = [
    "GZ01", "GZ02", "GZ06", "GZ09", "GZ13", "GZ17", "GZ18",
    *[f"LA{i:02d}" for i in range(1, 19)],
    *[f"MG{i:02d}" for i in range(4, 8)],
]

YEAR_MIN = 2009
YEAR_MAX = 2026
START_DATE = pd.Timestamp(f"{YEAR_MIN}-01-01")

# Thread workers are compatible with Jupyter/macOS and avoid multiprocessing
# pickling errors. Reduce to 1 if the external drives struggle with concurrent reads.
MAX_WORKERS = 5

SNR_SOURCES = [
    {
        "name": "cornice_refl_code",
        "root": Path(
            "/Volumes/cornice/marquardt/data/GNSS/docker/local/refl_code"
        ),
        "layout": "year_snr_station",
        "priority": 0,
    },
    {
        "name": "icefall_gnss_data",
        "root": Path("/Volumes/icefall/marquardt/data/GNSS_data"),
        "layout": "year_station_snr",
        "priority": 1,
    },
]

# New output folder. Existing station results are not overwritten unless
# OVERWRITE_EXISTING_STATIONS is True.
OUT_ROOT = Path(
    "/Volumes/cornice/marquardt/data/GNSS/GNSSAdjusted/"
    "multi_station_classified_processing"
)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

OVERWRITE_EXISTING_STATIONS = False

AZIMUTH_BINS = [
    (0, 60),
    (60, 120),
    (120, 180),
    (180, 240),
    (240, 300),
    (300, 360),
]

# Elevation and arc construction
E_MIN = 5.0
E_MAX = 25.0
MIN_POINTS = 25
MAX_TIME_GAP_SECONDS = 300.0
MIN_DIRECTION_STEP_DEG = 1.0e-4

# GNSS wavelength and RH search
L1_WAVELENGTH = 0.1902936
CF = L1_WAVELENGTH / 2.0

MIN_HEIGHT = CF
MAX_HEIGHT = 10.0
HEIGHT_STEP = 0.01
RH_SEARCH_GRID = np.arange(
    MIN_HEIGHT,
    MAX_HEIGHT + HEIGHT_STEP,
    HEIGHT_STEP,
)

# Quality thresholds
MIN_AMP = 15.0
MIN_PEAK_RATIO = 2.0
MIN_PEAK_BACKGROUND_RATIO = 20.0
MIN_INTERPRETABLE_HEIGHT = 0.45
MIN_INTERPRETABLE_CYCLES = 1.5
BOUNDARY_MARGIN_M = 0.05
MIN_PEAK_SEPARATION_M = 0.15
DAILY_MEDIAN_TOLERANCE_M = 0.30
MIN_FINAL_ARCS_PER_DAY = 5
MIN_RETAINED_FRACTION = 0.25

PLOT_DPI = 200

# =====================================================================
# FILE DISCOVERY
# =====================================================================


def parse_date_from_filename(filename: str) -> dt.date | None:
    match = re.search(r"(\d{3})0\.(\d{2})", filename.lower())
    if match is None:
        return None

    doy = int(match.group(1))
    year = 2000 + int(match.group(2))

    try:
        return dt.date(year, 1, 1) + dt.timedelta(days=doy - 1)
    except ValueError:
        return None


def extension_priority(filename: str) -> int:
    lower = filename.lower()
    if lower.endswith(".snr"):
        return 0
    if lower.endswith(".snr66"):
        return 1
    return 99


def station_snr_directory(source: dict, station: str, year: int) -> Path:
    station_lower = station.lower()

    if source["layout"] == "year_snr_station":
        return source["root"] / str(year) / "snr" / station_lower

    if source["layout"] == "year_station_snr":
        return source["root"] / str(year) / station_lower / "snr"

    raise ValueError(f"Unknown source layout: {source['layout']}")


def discover_unique_daily_files(station: str) -> list[dict]:
    candidates: list[dict] = []

    for year in range(YEAR_MIN, YEAR_MAX + 1):
        for source in SNR_SOURCES:
            snr_dir = station_snr_directory(source, station, year)

            if not snr_dir.is_dir():
                continue

            for path in sorted(snr_dir.iterdir()):
                if path.name.startswith("."):
                    continue

                if not path.name.lower().endswith((".snr", ".snr66")):
                    continue

                date = parse_date_from_filename(path.name)

                if date is None or date.year != year:
                    continue

                if pd.Timestamp(date) < START_DATE:
                    continue

                candidates.append(
                    {
                        "date": date,
                        "path": path,
                        "filename": path.name,
                        "source_name": source["name"],
                        "source_priority": source["priority"],
                        "extension_priority": extension_priority(path.name),
                    }
                )

    candidates.sort(
        key=lambda item: (
            item["date"],
            item["source_priority"],
            item["extension_priority"],
            item["filename"].lower(),
        )
    )

    selected: dict[dt.date, dict] = {}
    for item in candidates:
        selected.setdefault(item["date"], item)

    return [selected[date] for date in sorted(selected)]


# =====================================================================
# ARC SEPARATION
# =====================================================================


def split_into_continuous_directional_arcs(
    rows: np.ndarray,
) -> list[tuple[str, np.ndarray]]:
    if rows.shape[1] <= 3:
        return [("unknown", rows)]

    rows = rows[np.argsort(rows[:, 3])]
    time = rows[:, 3]
    elevation = rows[:, 1]

    if not np.all(np.isfinite(time)):
        return [("unknown", rows)]

    breaks = [0]
    previous_direction = 0

    for index in range(1, len(rows)):
        time_gap = time[index] - time[index - 1]
        elevation_step = elevation[index] - elevation[index - 1]

        if elevation_step > MIN_DIRECTION_STEP_DEG:
            direction = 1
        elif elevation_step < -MIN_DIRECTION_STEP_DEG:
            direction = -1
        else:
            direction = previous_direction

        direction_reversal = (
            previous_direction != 0
            and direction != 0
            and direction != previous_direction
        )

        if (
            time_gap <= 0
            or time_gap > MAX_TIME_GAP_SECONDS
            or direction_reversal
        ):
            breaks.append(index)
            previous_direction = 0
        elif direction != 0:
            previous_direction = direction

    breaks.append(len(rows))

    arcs: list[tuple[str, np.ndarray]] = []

    for start, stop in zip(breaks[:-1], breaks[1:]):
        arc = rows[start:stop]

        if len(arc) < 2:
            continue

        net_change = arc[-1, 1] - arc[0, 1]

        if net_change > 0:
            label = "rising"
        elif net_change < 0:
            label = "setting"
        else:
            label = "unknown"

        arcs.append((label, arc))

    return arcs


# =====================================================================
# PERIODOGRAM PEAKS AND ARC RETRIEVAL
# =====================================================================


def separated_peak_indices(power: np.ndarray) -> list[int]:
    finite_power = np.nan_to_num(
        power,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    grid_spacing = float(np.median(np.diff(RH_SEARCH_GRID)))
    minimum_distance_indices = max(
        1,
        int(round(MIN_PEAK_SEPARATION_M / grid_spacing)),
    )

    peak_indices, _ = find_peaks(
        finite_power,
        distance=minimum_distance_indices,
    )

    global_maximum = int(np.argmax(finite_power))

    if global_maximum not in peak_indices:
        peak_indices = np.append(peak_indices, global_maximum)

    ordered = peak_indices[
        np.argsort(finite_power[peak_indices])[::-1]
    ]

    return [int(index) for index in ordered]


def classify_preliminary_arc(
    retrieved_height: float,
    estimated_cycles: float,
    strongest_power: float,
    peak_ratio: float,
    peak_background_ratio: float,
) -> tuple[dict[str, bool], list[str]]:
    flags = {
        "fails_min_amp": (
            not np.isfinite(strongest_power)
            or strongest_power <= MIN_AMP
        ),
        "lower_search_boundary": (
            retrieved_height <= MIN_HEIGHT + BOUNDARY_MARGIN_M
        ),
        "upper_search_boundary": (
            retrieved_height >= MAX_HEIGHT - BOUNDARY_MARGIN_M
        ),
        "ambiguous_peak": (
            not np.isfinite(peak_ratio)
            or peak_ratio < MIN_PEAK_RATIO
        ),
        "weak_peak_background": (
            not np.isfinite(peak_background_ratio)
            or peak_background_ratio < MIN_PEAK_BACKGROUND_RATIO
        ),
        "below_interpretable_height": (
            retrieved_height < MIN_INTERPRETABLE_HEIGHT
        ),
        "below_interpretable_cycles": (
            estimated_cycles < MIN_INTERPRETABLE_CYCLES
        ),
    }

    reasons = [
        name
        for name, active in flags.items()
        if active
    ]

    return flags, reasons


def analyze_arc(
    station: str,
    arc: np.ndarray,
    date: dt.date,
    file_item: dict,
    satellite: int,
    direction: str,
    azimuth_min: float,
    azimuth_max: float,
    arc_number: int,
) -> dict | None:
    if len(arc) < MIN_POINTS:
        return None

    elevation = arc[:, 1].astype(float)
    azimuth = arc[:, 2].astype(float)
    linear_snr = 10.0 ** (arc[:, 6].astype(float) / 20.0)

    finite = (
        np.isfinite(elevation)
        & np.isfinite(azimuth)
        & np.isfinite(linear_snr)
    )

    elevation = elevation[finite]
    azimuth = azimuth[finite]
    linear_snr = linear_snr[finite]
    filtered_arc = arc[finite]

    if len(elevation) < MIN_POINTS:
        return None

    try:
        polynomial = np.polyfit(elevation, linear_snr, 3)
        trend = np.polyval(polynomial, elevation)
    except (TypeError, ValueError, np.linalg.LinAlgError):
        return None

    residual_snr = linear_snr - trend
    sin_elevation = np.sin(np.radians(elevation))

    order = np.argsort(sin_elevation)
    x_periodogram = sin_elevation[order] / CF
    y_periodogram = residual_snr[order]

    try:
        power = lombscargle(
            x_periodogram,
            y_periodogram,
            2.0 * np.pi * RH_SEARCH_GRID,
        )
    except (ValueError, ZeroDivisionError):
        return None

    if len(power) == 0 or not np.any(np.isfinite(power)):
        return None

    power = np.nan_to_num(
        power,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    peaks = separated_peak_indices(power)
    if not peaks:
        return None

    primary_index = peaks[0]
    secondary_index = peaks[1] if len(peaks) > 1 else None
    tertiary_index = peaks[2] if len(peaks) > 2 else None

    retrieved_height = float(RH_SEARCH_GRID[primary_index])
    strongest_power = float(power[primary_index])

    second_power = (
        float(power[secondary_index])
        if secondary_index is not None
        else np.nan
    )
    second_height = (
        float(RH_SEARCH_GRID[secondary_index])
        if secondary_index is not None
        else np.nan
    )

    third_power = (
        float(power[tertiary_index])
        if tertiary_index is not None
        else np.nan
    )
    third_height = (
        float(RH_SEARCH_GRID[tertiary_index])
        if tertiary_index is not None
        else np.nan
    )

    peak_ratio = (
        strongest_power / second_power
        if np.isfinite(second_power) and second_power > 0
        else np.nan
    )

    periodogram_background = float(np.median(power))
    peak_background_ratio = (
        strongest_power / periodogram_background
        if periodogram_background > 0
        else np.nan
    )

    elevation_min = float(np.min(elevation))
    elevation_max = float(np.max(elevation))

    sine_span = float(
        np.sin(np.radians(elevation_max))
        - np.sin(np.radians(elevation_min))
    )

    estimated_cycles = retrieved_height * sine_span / CF

    flags, reasons = classify_preliminary_arc(
        retrieved_height=retrieved_height,
        estimated_cycles=estimated_cycles,
        strongest_power=strongest_power,
        peak_ratio=peak_ratio,
        peak_background_ratio=peak_background_ratio,
    )

    preliminary_valid = len(reasons) == 0

    time_start = (
        float(np.min(filtered_arc[:, 3]))
        if filtered_arc.shape[1] > 3
        else np.nan
    )
    time_end = (
        float(np.max(filtered_arc[:, 3]))
        if filtered_arc.shape[1] > 3
        else np.nan
    )

    return {
        "Date": pd.Timestamp(date),
        "station": station,
        "year": date.year,
        "snr_file": file_item["filename"],
        "snr_path": str(file_item["path"]),
        "source_name": file_item["source_name"],
        "satellite": int(satellite),
        "direction": direction,
        "arc_number": int(arc_number),
        "azimuth_bin_min_deg": azimuth_min,
        "azimuth_bin_max_deg": azimuth_max,
        "azimuth_actual_min_deg": float(np.min(azimuth)),
        "azimuth_actual_max_deg": float(np.max(azimuth)),
        "time_start_seconds": time_start,
        "time_end_seconds": time_end,
        "n_points": int(len(elevation)),
        "elevation_min_deg": elevation_min,
        "elevation_max_deg": elevation_max,
        "elevation_span_deg": elevation_max - elevation_min,
        "retrieved_height_m": retrieved_height,
        "estimated_cycles": estimated_cycles,
        "max_power": strongest_power,
        "second_peak_height_m": second_height,
        "second_peak_power": second_power,
        "third_peak_height_m": third_height,
        "third_peak_power": third_power,
        "peak_to_second_ratio": peak_ratio,
        "periodogram_background": periodogram_background,
        "peak_to_background_ratio": peak_background_ratio,
        "secondary_relative_power": (
            second_power / strongest_power
            if np.isfinite(second_power) and strongest_power > 0
            else np.nan
        ),
        **flags,
        "preliminary_valid": preliminary_valid,
        "preliminary_rejection_reasons": ";".join(reasons),
        "daily_consensus_height_m": np.nan,
        "daily_consensus_difference_m": np.nan,
        "fails_daily_consensus": False,
        "final_valid": False,
        "final_rejection_reasons": "",
    }


# =====================================================================
# DAILY PROCESSING
# =====================================================================


def process_one_daily_file(
    station: str,
    file_item: dict,
) -> list[dict]:
    path = file_item["path"]
    date = file_item["date"]

    try:
        data = np.loadtxt(path)
    except Exception as exc:
        print(f"[{station}] WARN could not load {path}: {exc}")
        return []

    if data.ndim == 1:
        data = data.reshape(1, -1)

    if data.shape[1] < 7:
        print(f"[{station}] WARN fewer than 7 columns: {path}")
        return []

    records: list[dict] = []

    satellites = np.unique(
        data[:, 0][np.isfinite(data[:, 0])]
    ).astype(int)

    for azimuth_min, azimuth_max in AZIMUTH_BINS:
        for satellite in satellites:
            selection = (
                (data[:, 1] > E_MIN)
                & (data[:, 1] < E_MAX)
                & (data[:, 0] == satellite)
                & (data[:, 2] > azimuth_min)
                & (data[:, 2] < azimuth_max)
            )

            selected = data[selection]

            if len(selected) < MIN_POINTS:
                continue

            arcs = split_into_continuous_directional_arcs(selected)

            for arc_number, (direction, arc) in enumerate(arcs, start=1):
                record = analyze_arc(
                    station=station,
                    arc=arc,
                    date=date,
                    file_item=file_item,
                    satellite=satellite,
                    direction=direction,
                    azimuth_min=azimuth_min,
                    azimuth_max=azimuth_max,
                    arc_number=arc_number,
                )

                if record is not None:
                    records.append(record)

    return records


# =====================================================================
# CLASSIFICATION AND DAILY CONSENSUS
# =====================================================================


def apply_daily_consensus_filter(
    arc_df: pd.DataFrame,
) -> pd.DataFrame:
    output = arc_df.copy()

    preliminary = output[output["preliminary_valid"]].copy()

    daily_consensus = (
        preliminary.groupby("Date")["retrieved_height_m"]
        .median()
        .rename("daily_consensus_height_m")
    )

    output = output.drop(
        columns=["daily_consensus_height_m"]
    ).merge(
        daily_consensus,
        on="Date",
        how="left",
    )

    output["daily_consensus_difference_m"] = (
        output["retrieved_height_m"]
        - output["daily_consensus_height_m"]
    ).abs()

    output["fails_daily_consensus"] = (
        output["preliminary_valid"]
        & (
            output["daily_consensus_difference_m"]
            > DAILY_MEDIAN_TOLERANCE_M
        )
    )

    output["final_valid"] = (
        output["preliminary_valid"]
        & ~output["fails_daily_consensus"]
    )

    def final_reason(row: pd.Series) -> str:
        reasons = []

        preliminary_reason = str(
            row["preliminary_rejection_reasons"]
        ).strip()

        if preliminary_reason:
            reasons.extend(
                reason
                for reason in preliminary_reason.split(";")
                if reason
            )

        if bool(row["fails_daily_consensus"]):
            reasons.append("fails_daily_consensus")

        return ";".join(reasons)

    output["final_rejection_reasons"] = output.apply(
        final_reason,
        axis=1,
    )

    return output


def build_daily_summary(
    station: str,
    arc_df: pd.DataFrame,
) -> pd.DataFrame:
    all_power = arc_df[~arc_df["fails_min_amp"]].copy()
    preliminary = arc_df[arc_df["preliminary_valid"]].copy()
    final = arc_df[arc_df["final_valid"]].copy()

    all_summary = (
        all_power.groupby("Date", as_index=False)
        .agg(
            RH_median_all_power_m=("retrieved_height_m", "median"),
            RH_mean_all_power_m=("retrieved_height_m", "mean"),
            RH_N_all_power=("retrieved_height_m", "count"),
            ambiguous_fraction=("ambiguous_peak", "mean"),
            weak_background_fraction=("weak_peak_background", "mean"),
            low_height_fraction=("below_interpretable_height", "mean"),
            low_cycle_fraction=("below_interpretable_cycles", "mean"),
            upper_boundary_fraction=("upper_search_boundary", "mean"),
            peak_ratio_median=("peak_to_second_ratio", "median"),
            peak_background_ratio_median=(
                "peak_to_background_ratio",
                "median",
            ),
        )
    )

    preliminary_summary = (
        preliminary.groupby("Date", as_index=False)
        .agg(
            RH_median_preliminary_m=("retrieved_height_m", "median"),
            RH_mean_preliminary_m=("retrieved_height_m", "mean"),
            RH_N_preliminary=("retrieved_height_m", "count"),
        )
    )

    final_summary = (
        final.groupby("Date", as_index=False)
        .agg(
            RH_median_filtered_m=("retrieved_height_m", "median"),
            RH_mean_filtered_m=("retrieved_height_m", "mean"),
            RH_std_filtered_m=("retrieved_height_m", "std"),
            RH_N_filtered=("retrieved_height_m", "count"),
        )
    )

    summary = (
        all_summary
        .merge(preliminary_summary, on="Date", how="left")
        .merge(final_summary, on="Date", how="left")
    )

    summary["retained_fraction_of_all_power"] = (
        summary["RH_N_filtered"]
        / summary["RH_N_all_power"].replace(0, np.nan)
    )

    summary["passes_minimum_arc_count"] = (
        summary["RH_N_filtered"].fillna(0)
        >= MIN_FINAL_ARCS_PER_DAY
    )

    summary["passes_retained_fraction"] = (
        summary["retained_fraction_of_all_power"].fillna(0)
        >= MIN_RETAINED_FRACTION
    )

    summary["daily_filtered_valid"] = (
        summary["passes_minimum_arc_count"]
        & summary["passes_retained_fraction"]
    )

    summary["RH_surface_height_m"] = (
        summary["RH_median_filtered_m"]
        .where(summary["daily_filtered_valid"])
    )

    summary["daily_rejection_reason"] = np.select(
        [
            ~summary["passes_minimum_arc_count"],
            (
                summary["passes_minimum_arc_count"]
                & ~summary["passes_retained_fraction"]
            ),
        ],
        [
            "too_few_filtered_arcs",
            "retained_fraction_below_0.25",
        ],
        default="",
    )

    summary.insert(1, "station", station)
    summary["year"] = summary["Date"].dt.year

    return summary.sort_values("Date")


# =====================================================================
# OUTPUT
# =====================================================================


def save_comparison_plot(
    station: str,
    daily_df: pd.DataFrame,
    station_dir: Path,
) -> Path:
    output = station_dir / f"{station.lower()}_filtered_comparison.png"

    fig, ax = plt.subplots(figsize=(14, 7))

    ax.scatter(
        daily_df["Date"],
        daily_df["RH_median_all_power_m"],
        s=10,
        alpha=0.30,
        label="Daily median: all power-accepted arcs",
    )

    ax.scatter(
        daily_df["Date"],
        daily_df["RH_median_filtered_m"],
        s=11,
        alpha=0.30,
        label="Diagnostic median: classified + consensus filtered",
    )

    ax.scatter(
        daily_df["Date"],
        daily_df["RH_surface_height_m"],
        s=16,
        label="Final RH: retained fraction >= 2filtered",
    )

    ax.set_xlabel("Date")
    ax.set_ylabel("Reflector height (m)")
    ax.grid(True, alpha=0.3)

    ax2 = ax.twinx()
    ax2.plot(
        daily_df["Date"],
        daily_df["retained_fraction_of_all_power"],
        linewidth=1.0,
        alpha=0.70,
        label="Retained arc fraction",
    )
    ax2.set_ylabel("Fraction of power-accepted arcs retained")
    ax2.set_ylim(-0.02, 1.02)

    handles_1, labels_1 = ax.get_legend_handles_labels()
    handles_2, labels_2 = ax2.get_legend_handles_labels()

    ax.legend(
        handles_1 + handles_2,
        labels_1 + labels_2,
        loc="best",
    )

    ax.set_title(f"{station} GNSS-IR raw and quality-classified RH")

    fig.tight_layout()
    fig.savefig(output, dpi=PLOT_DPI)
    plt.close(fig)

    return output


def process_station(station: str) -> dict:
    station = station.upper()
    station_dir = OUT_ROOT / station.lower()
    station_dir.mkdir(parents=True, exist_ok=True)

    daily_csv = station_dir / f"{station.lower()}_daily_summary_classified.csv"

    if daily_csv.is_file() and not OVERWRITE_EXISTING_STATIONS:
        print(f"[{station}] SKIP existing output: {daily_csv}")
        return {
            "station": station,
            "status": "already_done",
            "daily_csv": str(daily_csv),
            "arc_rows": np.nan,
            "daily_rows": np.nan,
            "valid_daily_rows": np.nan,
        }

    files = discover_unique_daily_files(station)

    if not files:
        print(f"[{station}] No SNR files found.")
        return {
            "station": station,
            "status": "no_files",
            "daily_csv": None,
            "arc_rows": 0,
            "daily_rows": 0,
            "valid_daily_rows": 0,
        }

    print(f"[{station}] Processing {len(files)} unique dates.")

    all_records: list[dict] = []

    for index, file_item in enumerate(files, start=1):
        if index == 1 or index % 100 == 0 or index == len(files):
            print(
                f"[{station}] {index}/{len(files)} "
                f"{file_item['date']}"
            )

        all_records.extend(
            process_one_daily_file(station, file_item)
        )

    arc_df = pd.DataFrame(all_records)

    if arc_df.empty:
        print(f"[{station}] No arcs could be processed.")
        return {
            "station": station,
            "status": "no_arcs",
            "daily_csv": None,
            "arc_rows": 0,
            "daily_rows": 0,
            "valid_daily_rows": 0,
        }

    arc_df = arc_df.sort_values(
        [
            "Date",
            "satellite",
            "direction",
            "azimuth_bin_min_deg",
            "arc_number",
        ]
    ).reset_index(drop=True)

    classified_df = apply_daily_consensus_filter(arc_df)
    daily_df = build_daily_summary(station, classified_df)

    all_csv = station_dir / f"{station.lower()}_arc_retrievals_all.csv"
    valid_csv = station_dir / f"{station.lower()}_arc_retrievals_valid.csv"
    rejected_csv = station_dir / f"{station.lower()}_arc_retrievals_rejected.csv"

    classified_df.to_csv(all_csv, index=False)
    classified_df[classified_df["final_valid"]].to_csv(valid_csv, index=False)
    classified_df[~classified_df["final_valid"]].to_csv(rejected_csv, index=False)
    daily_df.to_csv(daily_csv, index=False)

    plot_path = save_comparison_plot(
        station,
        daily_df,
        station_dir,
    )

    valid_daily_rows = int(daily_df["daily_filtered_valid"].sum())

    print(
        f"[{station}] COMPLETE: arcs={len(classified_df)}, "
        f"days={len(daily_df)}, valid_days={valid_daily_rows}"
    )

    return {
        "station": station,
        "status": "ok",
        "daily_csv": str(daily_csv),
        "plot": str(plot_path),
        "arc_rows": len(classified_df),
        "daily_rows": len(daily_df),
        "valid_daily_rows": valid_daily_rows,
        "date_min": daily_df["Date"].min(),
        "date_max": daily_df["Date"].max(),
    }


# =====================================================================
# MAIN
# =====================================================================


def main() -> None:
    print("========== MULTI-STATION GNSS-IR PROCESSING ==========")
    print(f"Stations requested: {len(STATIONS)}")
    print(f"Years: {YEAR_MIN}-{YEAR_MAX}")
    print(f"Output root: {OUT_ROOT}")
    print(f"Workers: {MAX_WORKERS}")
    print("")

    results: list[dict] = []

    if MAX_WORKERS == 1:
        for station in STATIONS:
            results.append(process_station(station))
    else:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_map = {
                executor.submit(process_station, station): station
                for station in STATIONS
            }

            for future in as_completed(future_map):
                station = future_map[future]

                try:
                    result = future.result()
                except Exception as exc:
                    print(f"[{station}] CRASH: {exc}")
                    result = {
                        "station": station,
                        "status": "crash",
                        "error": str(exc),
                    }

                results.append(result)

    results_df = pd.DataFrame(results).sort_values("station")
    processing_summary_csv = OUT_ROOT / "all_stations_processing_summary.csv"
    results_df.to_csv(processing_summary_csv, index=False)

    daily_frames: list[pd.DataFrame] = []

    for result in results:
        daily_path = result.get("daily_csv")

        if not daily_path:
            continue

        path = Path(daily_path)

        if path.is_file():
            frame = pd.read_csv(path)
            frame["Date"] = pd.to_datetime(frame["Date"], errors="coerce")
            daily_frames.append(frame)

    if daily_frames:
        combined_daily = pd.concat(
            daily_frames,
            ignore_index=True,
        ).sort_values(["station", "Date"])

        combined_daily_csv = (
            OUT_ROOT
            / "all_stations_daily_summary_classified.csv"
        )
        combined_daily.to_csv(combined_daily_csv, index=False)
    else:
        combined_daily_csv = None

    print("")
    print("========== COMPLETE ==========")
    print(f"Processing summary: {processing_summary_csv}")
    print(f"Combined daily CSV: {combined_daily_csv}")

    status_counts = results_df["status"].value_counts(dropna=False)
    print("")
    print(status_counts.to_string())


if __name__ == "__main__":
    main()


## Build corrected GNSS surface-elevation time series

In [ ]:
from pathlib import Path
import datetime as dt
import gzip
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# PATHS
# ============================================================

# Classified RH outputs produced by the new GNSS-IR processor.
RH_ROOT = Path(
    "/Volumes/cornice/marquardt/data/GNSS/GNSSAdjusted/"
    "multi_station_classified_processing"
)

# Daily GNSS position averages produced by
# average_daily_regular_gnss_positions_2011_2026.ipynb.
GNSS_DAILY_POSITION_DIR = Path(
    "/Volumes/icefall/tinydrive/GNSS_data/daily_positions"
)

RINEX_SOURCES = [
    {
        "name": "cornice_refl_code",
        "root": Path(
            "/Volumes/cornice/marquardt/data/GNSS/docker/local/refl_code"
        ),
        "layout": "year_rinex_station",
        "priority": 0,
    },
    {
        "name": "icefall_gnss_data",
        "root": Path(
            "/Volumes/icefall/marquardt/data/GNSS_data"
        ),
        "layout": "year_station_rinex",
        "priority": 1,
    },
    {
        "name": "cornice_docker_station_year",
        "root": Path(
            "/Volumes/cornice/marquardt/data/GNSS/docker"
        ),
        "layout": "station_year",
        "priority": 2,
    },
]

ATX_PATH = Path("/Volumes/cornice/marquardt/data/GNSS/igs20.atx")

# The first run on a given day uses YYYY-MM-DD exactly. If that folder already
# exists, _run_02, _run_03, etc. are appended so no prior output is overwritten.
OUTPUT_PARENT = Path(
    "/Volumes/cornice/marquardt/data/GNSS/GNSSAdjusted/"
    "classified_rh_apc_corrections"
)


def make_unique_dated_output_folder(parent: Path) -> Path:
    parent.mkdir(parents=True, exist_ok=True)
    date_name = dt.date.today().isoformat()
    candidate = parent / date_name

    if not candidate.exists():
        candidate.mkdir(parents=True)
        return candidate

    run_number = 2
    while True:
        candidate = parent / f"{date_name}_run_{run_number:02d}"
        if not candidate.exists():
            candidate.mkdir(parents=True)
            return candidate
        run_number += 1


OUT_DIR = make_unique_dated_output_folder(OUTPUT_PARENT)
APC_OUT_DIR = OUT_DIR / "corrected_timeseries"
PLOT_DIR = OUT_DIR / "plots"
DIAG_DIR = OUT_DIR / "diagnostics"

for folder in (APC_OUT_DIR, PLOT_DIR, DIAG_DIR):
    folder.mkdir(parents=True, exist_ok=False)

# ============================================================
# SETTINGS
# ============================================================

YEARS = list(range(2008, 2027))
START_DATE = pd.Timestamp("2011-01-01")

STATIONS_ONLY = [
    "GZ01", "GZ02", "GZ06", "GZ09", "GZ13", "GZ17", "GZ18",
    *[f"LA{i:02d}" for i in range(1, 19)],
    *[f"MG{i:02d}" for i in range(4, 8)],
]

N_CUTOFF_K = 3.0
N_ABS_MIN = None

ROLL_WINDOW_DAYS = 61
MAD_THRESH = 3.5
MIN_PERIODS = 10
RATE_THRESH_MPD = 0.5

ANTEX_URLS = [
    "https://files.igs.org/pub/station/general/igs20.atx",
    "https://igs.org/pub/station/general/igs20.atx",
]

EXCLUDE_BY_STATION = {
    "gz17": [("2012-07-22", "2013-01-30")],
    "mg04": [
        ("2015-09-11", "2015-09-11"),
        ("2015-09-12", "2015-09-12"),
        ("2015-09-13", "2015-09-13"),
        ("2015-09-14", "2015-09-14"),
    ],
}

FINAL_COLS_OUT = [
    "YEAR-MM-DD",
    "STN",
    "LATDD",
    "LONDD",
    "GNSS mean daily height",
    "GNSS std",
    "GNSS N",
    "RH_median_filtered_m",
    "RH_mean_filtered_m",
    "RH_std_filtered_m",
    "RH_N_filtered",
    "retained_fraction_of_all_power",
    "daily_filtered_valid",
    "GNSS_uncertainty_m",
    "RH_uncertainty_m",
    "Adjusted_original(m)",
    "Adjusted_original_uncertainty_m",
    "APC_correction_m",
    "Adjusted(m)",
    "Adjusted_uncertainty_m",
]


# ============================================================
# HELPERS
# ============================================================

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def rolling_hampel(series, window=61, min_periods=10):
    s = pd.Series(series, dtype="float64")
    med = s.rolling(window, center=True, min_periods=min_periods).median()
    mad = s.rolling(window, center=True, min_periods=min_periods).apply(
        lambda v: np.nanmedian(np.abs(v - np.nanmedian(v))),
        raw=False,
    )
    sigma = 1.4826 * mad
    return ((s - med) / sigma).to_numpy()


def download_antex_if_needed(atx_path: Path, urls: list[str]):
    if atx_path.exists() and atx_path.stat().st_size > 0:
        print(f"[ANTEX] Using existing ANTEX file: {atx_path}")
        return

    atx_path.parent.mkdir(parents=True, exist_ok=True)

    last_error = None

    for url in urls:
        try:
            print(f"[ANTEX] Downloading {url}")
            urllib.request.urlretrieve(url, atx_path)
            print(f"[ANTEX] Saved to {atx_path}")
            return
        except Exception as e:
            last_error = e
            print(f"  ⚠️ Failed: {e}")

    raise RuntimeError(f"Could not download ANTEX file. Last error: {last_error}")


# ============================================================
# GNSS DAILY LOADERS
# ============================================================

def standardize_station(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower()


def identify_date_column(columns) -> str | None:
    for candidate in [
        "YEAR-MM-DD",
        "date_only",
        "Date",
        "date",
        "mid_day",
        "datetime",
    ]:
        if candidate in columns:
            return candidate
    return None


def identify_gnss_std_column(columns) -> str | None:
    """
    Only accept the exact feature name requested by the user.
    """
    return "GNSS std" if "GNSS std" in columns else None


def load_station_daily_position_average(
    station: str,
    base_dir: Path,
) -> tuple[pd.DataFrame, dict]:
    """Load one station's daily averages from the regular GNSS POS output."""
    station_lower = station.lower()
    path = base_dir / f"{station_lower}_daily_mean_2011_2026.csv"

    if not path.is_file():
        return pd.DataFrame(), {
            "station": station.upper(),
            "file_found": False,
            "file_path": str(path),
            "rows_loaded": 0,
            "date_min": "",
            "date_max": "",
            "missing_columns": "",
            "status": "missing_file",
        }

    df = pd.read_csv(path)
    required = {
        "STN",
        "date",
        "lat_mean",
        "lon_mean",
        "h_mean",
        "h_std",
    }
    missing = required - set(df.columns)
    if missing:
        return pd.DataFrame(), {
            "station": station.upper(),
            "file_found": True,
            "file_path": str(path),
            "rows_loaded": 0,
            "date_min": "",
            "date_max": "",
            "missing_columns": ";".join(sorted(missing)),
            "status": "missing_required_columns",
        }

    # The averaging notebook's h_mean and h_std are the daily mean GNSS
    # elevation and its standard deviation used in the surface calculation.
    df = df.rename(columns={
        "date": "YEAR-MM-DD",
        "lat_mean": "LATDD",
        "lon_mean": "LONDD",
        "h_mean": "GNSS mean daily height",
        "h_std": "GNSS std",
        "count": "GNSS N",
    })
    df["YEAR-MM-DD"] = pd.to_datetime(
        df["YEAR-MM-DD"],
        errors="coerce",
    ).dt.normalize()
    df["date_only"] = df["YEAR-MM-DD"].dt.strftime("%Y-%m-%d")
    df["STN"] = df["STN"].map(standardize_station)
    df = df.loc[df["STN"].eq(station_lower)].copy()

    for column in [
        "LATDD",
        "LONDD",
        "GNSS mean daily height",
        "GNSS std",
        "GNSS N",
    ]:
        df[column] = safe_numeric(df[column])

    df["HGT(m)"] = np.nan
    df["gnss_source"] = "regular_daily_position_average"
    df["gnss_source_file"] = str(path)

    keep = [
        "YEAR-MM-DD",
        "date_only",
        "STN",
        "LATDD",
        "LONDD",
        "HGT(m)",
        "GNSS mean daily height",
        "GNSS std",
        "GNSS N",
        "gnss_source",
        "gnss_source_file",
    ]
    output = (
        df[keep]
        .dropna(subset=["YEAR-MM-DD", "GNSS mean daily height"])
        .drop_duplicates(subset=["STN", "YEAR-MM-DD"], keep="first")
        .sort_values("YEAR-MM-DD")
        .reset_index(drop=True)
    )

    return output, {
        "station": station.upper(),
        "file_found": True,
        "file_path": str(path),
        "rows_loaded": len(output),
        "date_min": output["YEAR-MM-DD"].min(),
        "date_max": output["YEAR-MM-DD"].max(),
        "missing_columns": "",
        "status": "ok" if not output.empty else "no_valid_rows",
    }


def load_all_gnss_daily(
    daily_position_dir: Path,
    stations: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load the daily regular-GNSS averages for the requested stations."""
    station_frames = []
    diagnostics = []

    for station in stations:
        frame, diagnostic = load_station_daily_position_average(
            station=station,
            base_dir=daily_position_dir,
        )
        diagnostics.append(diagnostic)
        if not frame.empty:
            station_frames.append(frame)

    if not station_frames:
        raise RuntimeError(
            "No daily regular-GNSS station files were loaded."
        )

    combined = (
        pd.concat(station_frames, ignore_index=True, sort=False)
        .sort_values(["STN", "YEAR-MM-DD"])
        .drop_duplicates(subset=["STN", "YEAR-MM-DD"], keep="first")
        .reset_index(drop=True)
    )
    return combined, pd.DataFrame(diagnostics)


def get_header_label(line: str) -> str:
    return line[60:].strip() if len(line) >= 61 else ""


def open_text_maybe_gzip(path: Path):
    if path.name.lower().endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8", errors="ignore")
    return path.open("r", encoding="utf-8", errors="ignore")


def read_rinex_header(path: Path) -> list[str]:
    header_lines = []

    with open_text_maybe_gzip(path) as f:
        for line in f:
            header_lines.append(line.rstrip("\n"))
            if "END OF HEADER" in line:
                break

    if not header_lines or "END OF HEADER" not in header_lines[-1]:
        raise ValueError(f"Could not find END OF HEADER in {path}")

    return header_lines


def extract_antenna_type(header_lines: list[str]):
    antenna_number = None
    antenna_type = None

    for line in header_lines:
        if get_header_label(line) == "ANT # / TYPE":
            antenna_number = line[:20].strip() or None
            antenna_type = line[20:40].strip() or None
            break

    return antenna_number, antenna_type


def extract_observation_types(header_lines: list[str]) -> list[str]:
    obs_types = []

    for line in header_lines:
        if get_header_label(line) == "# / TYPES OF OBSERV":
            parts = line[:60].split()
            if len(parts) > 1:
                obs_types.extend(parts[1:])

    return obs_types


def extract_signal_bands(obs_types: list[str]) -> list[str]:
    bands = set()

    for obs in obs_types:
        obs = str(obs).strip().upper()
        if len(obs) >= 2 and obs[1:].isdigit():
            bands.add(obs[1:])

    return sorted(bands, key=lambda x: int(x))


def station_rinex_directory(
    source: dict,
    station: str,
    year: int,
) -> Path:
    station_lower = station.lower()

    if source["layout"] == "year_rinex_station":
        return source["root"] / str(year) / "rinex" / station_lower

    if source["layout"] == "year_station_rinex":
        return source["root"] / str(year) / station_lower / "rinex"

    if source["layout"] == "station_year":
        return source["root"] / station_lower / str(year)

    raise ValueError(f"Unknown RINEX source layout: {source['layout']}")


def list_rinex_files(folder: Path) -> list[Path]:
    if not folder.exists():
        return []

    files = sorted(p for p in folder.iterdir() if p.is_file())
    rinex_like = []

    for p in files:
        # Skip macOS AppleDouble metadata files such as:
        #     ._la090010.11d
        # These are not real RINEX files and do not contain valid headers.
        if p.name.startswith("._"):
            continue

        name = p.name.lower()

        if (
            name.endswith((".d", ".o", ".crx", ".rnx", ".gz", ".z", ".zip"))
            or any(f".{yy:02d}d" in name for yy in range(8, 27))
            or any(f".{yy:02d}o" in name for yy in range(8, 27))
        ):
            rinex_like.append(p)

    return rinex_like


def build_rinex_antenna_table(stations: list[str]) -> pd.DataFrame:
    records = []

    for station in stations:
        station_upper = station.upper()

        for year in YEARS:
            candidates = []

            for source in RINEX_SOURCES:
                rinex_dir = station_rinex_directory(
                    source=source,
                    station=station,
                    year=year,
                )

                rinex_files = list_rinex_files(rinex_dir)

                for path in rinex_files:
                    candidates.append({
                        "path": path,
                        "rinex_dir": rinex_dir,
                        "source_name": source["name"],
                        "source_priority": source["priority"],
                    })

            candidates.sort(
                key=lambda item: (
                    item["source_priority"],
                    item["path"].name.lower(),
                )
            )

            if not candidates:
                searched_dirs = [
                    str(
                        station_rinex_directory(
                            source=source,
                            station=station,
                            year=year,
                        )
                    )
                    for source in RINEX_SOURCES
                ]

                records.append({
                    "station": station_upper,
                    "year": year,
                    "rinex_dir": "",
                    "searched_rinex_dirs": " | ".join(searched_dirs),
                    "source_name": None,
                    "file_found": False,
                    "first_file": None,
                    "antenna_number": None,
                    "antenna_type": None,
                    "obs_types": None,
                    "bands_used": None,
                    "has_L1": False,
                    "has_L2": False,
                    "dual_freq_L1L2": False,
                    "read_status": "no_files",
                })
                continue

            selected = candidates[0]
            first_file = selected["path"]

            try:
                header_lines = read_rinex_header(first_file)
                antenna_number, antenna_type = extract_antenna_type(header_lines)
                obs_types = extract_observation_types(header_lines)
                bands_used = extract_signal_bands(obs_types)

                has_L1 = "L1" in obs_types
                has_L2 = "L2" in obs_types

                records.append({
                    "station": station_upper,
                    "year": year,
                    "rinex_dir": str(selected["rinex_dir"]),
                    "searched_rinex_dirs": "",
                    "source_name": selected["source_name"],
                    "file_found": True,
                    "first_file": first_file.name,
                    "antenna_number": antenna_number,
                    "antenna_type": antenna_type,
                    "obs_types": ", ".join(obs_types) if obs_types else None,
                    "bands_used": ", ".join(bands_used) if bands_used else None,
                    "has_L1": has_L1,
                    "has_L2": has_L2,
                    "dual_freq_L1L2": has_L1 and has_L2,
                    "read_status": "ok",
                })

            except Exception as e:
                records.append({
                    "station": station_upper,
                    "year": year,
                    "rinex_dir": str(selected["rinex_dir"]),
                    "searched_rinex_dirs": "",
                    "source_name": selected["source_name"],
                    "file_found": True,
                    "first_file": first_file.name,
                    "antenna_number": None,
                    "antenna_type": None,
                    "obs_types": None,
                    "bands_used": None,
                    "has_L1": False,
                    "has_L2": False,
                    "dual_freq_L1L2": False,
                    "read_status": f"error: {e}",
                })

    return pd.DataFrame(records)



def summarize_rinex_coverage(rinex_df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize RINEX availability by station across the configured YEARS.

    A station is classified as:
      - complete: RINEX file found and read successfully for every year
      - partial: at least one year found, at least one year missing or errored
      - missing_all_years: no readable RINEX file found for any year
    """
    rows = []

    for station, group in rinex_df.groupby("station", sort=True):
        group = group.sort_values("year").copy()

        ok_mask = group["read_status"].eq("ok")
        no_file_mask = group["read_status"].eq("no_files")
        error_mask = ~(ok_mask | no_file_mask)

        years_found = group.loc[ok_mask, "year"].astype(int).tolist()
        years_missing = group.loc[no_file_mask, "year"].astype(int).tolist()
        years_error = group.loc[error_mask, "year"].astype(int).tolist()

        if len(years_found) == len(group):
            coverage_status = "complete"
        elif len(years_found) == 0:
            coverage_status = "missing_all_years"
        else:
            coverage_status = "partial"

        rows.append({
            "station": station,
            "coverage_status": coverage_status,
            "n_years_requested": len(group),
            "n_years_found": len(years_found),
            "n_years_missing": len(years_missing),
            "n_years_error": len(years_error),
            "years_found": ",".join(str(year) for year in years_found),
            "years_missing": ",".join(str(year) for year in years_missing),
            "years_error": ",".join(str(year) for year in years_error),
        })

    return pd.DataFrame(rows)


def print_rinex_coverage_report(rinex_summary: pd.DataFrame) -> None:
    missing_any = rinex_summary[
        rinex_summary["coverage_status"] != "complete"
    ].copy()

    print("\n[RINEX] Station coverage summary:")
    print(
        rinex_summary["coverage_status"]
        .value_counts(dropna=False)
        .to_string()
    )

    if missing_any.empty:
        print("[RINEX] Every requested station has readable RINEX files for all years.")
        return

    print("\n[RINEX] Stations with missing or unreadable RINEX years:")

    for _, row in missing_any.sort_values("station").iterrows():
        station = row["station"]
        status = row["coverage_status"]
        found = int(row["n_years_found"])
        requested = int(row["n_years_requested"])
        missing_years = row["years_missing"]
        error_years = row["years_error"]

        print(f"  {station}: {status}, found {found}/{requested} years")

        if missing_years:
            print(f"    missing years: {missing_years}")

        if error_years:
            print(f"    unreadable/error years: {error_years}")


# ============================================================
# ANTEX PARSING + L1 APC TABLE
# ============================================================

def parse_antex_blocks(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(f"ANTEX file does not exist: {path}")

    with path.open("r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    blocks = []
    current_block = None
    current_freq = None

    for line in lines:
        label = line[60:].strip() if len(line) >= 61 else ""

        if label == "START OF ANTENNA":
            current_block = {
                "antenna_type_full": None,
                "antenna_type": None,
                "radome": None,
                "freq_offsets": {},
            }
            current_freq = None
            continue

        if current_block is None:
            continue

        if label == "TYPE / SERIAL NO":
            ant_full = line[:20].strip()
            current_block["antenna_type_full"] = ant_full

            parts = ant_full.split()
            current_block["antenna_type"] = parts[0] if parts else None
            current_block["radome"] = parts[1] if len(parts) >= 2 else None

        elif label == "START OF FREQUENCY":
            current_freq = line[:10].strip()

        elif label == "NORTH / EAST / UP" and current_freq is not None:
            parts = line[:60].split()
            if len(parts) >= 3:
                current_block["freq_offsets"][current_freq] = {
                    "north_mm": float(parts[0]),
                    "east_mm": float(parts[1]),
                    "up_mm": float(parts[2]),
                }

        elif label == "END OF FREQUENCY":
            current_freq = None

        elif label == "END OF ANTENNA":
            blocks.append(current_block)
            current_block = None
            current_freq = None

    return blocks


def normalize_antenna_type(rinex_ant):
    if rinex_ant is None or pd.isna(rinex_ant):
        return None, None

    parts = str(rinex_ant).strip().split()

    if len(parts) >= 2:
        return parts[0], parts[1]
    if len(parts) == 1:
        return parts[0], None

    return None, None


def find_matching_antex_block(blocks: list[dict], rinex_antenna_type):
    ant_type, radome = normalize_antenna_type(rinex_antenna_type)

    if ant_type is None:
        return None

    exact = [
        b for b in blocks
        if b["antenna_type"] == ant_type and b["radome"] == radome
    ]
    if exact:
        return exact[0]

    type_only = [
        b for b in blocks
        if b["antenna_type"] == ant_type
    ]
    if type_only:
        return type_only[0]

    return None


def build_l1_offset_table(rinex_df: pd.DataFrame, atx_path: Path) -> pd.DataFrame:
    blocks = parse_antex_blocks(atx_path)
    records = []

    for _, row in rinex_df.iterrows():
        rinex_ant = row.get("antenna_type", None)
        block = find_matching_antex_block(blocks, rinex_ant)

        if block is None:
            records.append({
                "station": row["station"],
                "year": row["year"],
                "rinex_antenna_type": rinex_ant,
                "antex_match": None,
                "gps_l1_up_mm": None,
                "gps_l1_up_m": None,
                "offset_status": "no_antex_match",
            })
            continue

        g01 = block["freq_offsets"].get("G01", {})
        g01_up_mm = g01.get("up_mm", None)

        records.append({
            "station": row["station"],
            "year": row["year"],
            "rinex_antenna_type": rinex_ant,
            "antex_match": block["antenna_type_full"],
            "gps_l1_up_mm": g01_up_mm,
            "gps_l1_up_m": g01_up_mm / 1000.0 if g01_up_mm is not None else None,
            "offset_status": "ok" if g01_up_mm is not None else "missing_g01",
        })

    return (
        pd.DataFrame(records)
        .sort_values(["station", "year"])
        .reset_index(drop=True)
    )


# ============================================================
# CLASSIFIED RH LOADERS
# ============================================================

def discover_rh_files_for_stations(
    rh_root: Path,
    stations: list[str],
) -> dict[str, list[Path]]:
    """
    Locate the new per-station classified daily summary:
        RH_ROOT/<station>/<station>_daily_summary_classified.csv
    """
    rh_files: dict[str, list[Path]] = {}

    for station in stations:
        st = station.lower()
        path = rh_root / st / f"{st}_daily_summary_classified.csv"

        if path.is_file() and not path.name.startswith("._"):
            rh_files[st] = [path]

    return rh_files


def load_rh_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    required = {
        "Date",
        "station",
        "RH_surface_height_m",
    }

    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} missing classified RH columns: {missing}")

    df = df.copy()
    df["YEAR-MM-DD"] = pd.to_datetime(df["Date"], errors="coerce")
    df["STN"] = df["station"].astype(str).str.strip().str.lower()

    numeric_cols = [
        "RH_surface_height_m",
        "RH_median_filtered_m",
        "RH_mean_filtered_m",
        "RH_std_filtered_m",
        "RH_N_filtered",
        "retained_fraction_of_all_power",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = safe_numeric(df[col])

    if "daily_filtered_valid" in df.columns:
        df["daily_filtered_valid"] = (
            df["daily_filtered_valid"]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({"true": True, "false": False})
        )
    else:
        df["daily_filtered_valid"] = df["RH_surface_height_m"].notna()

    # Downstream compatibility: the original correction code used this name.
    df["RH median daily height"] = df["RH_surface_height_m"]

    # Only final accepted daily RH values are eligible for the correction.
    df = df.dropna(
        subset=["YEAR-MM-DD", "STN", "RH_surface_height_m"]
    ).copy()

    df["rh_source_file"] = str(path)
    return df


def load_all_rh(rh_files: dict[str, list[Path]]) -> pd.DataFrame:
    frames = []

    for st, paths in rh_files.items():
        for path in paths:
            try:
                frames.append(load_rh_file(path))
            except Exception as e:
                print(f"[RH] Failed {path}: {e}")

    if not frames:
        raise RuntimeError("No classified RH daily files loaded.")

    return (
        pd.concat(frames, ignore_index=True)
        .sort_values(["STN", "YEAR-MM-DD"])
        .drop_duplicates(subset=["STN", "YEAR-MM-DD"], keep="first")
        .reset_index(drop=True)
    )


# ============================================================
# FILTERING + APC
# ============================================================

def compute_global_lowN_threshold(station_dfs, start_date, k=3.0, abs_min=None):
    pooled = []

    for _, df in station_dfs.items():
        x = df[df["YEAR-MM-DD"] >= start_date].copy()
        x["GNSS N"] = safe_numeric(x["GNSS N"])
        x = x.dropna(subset=["GNSS N"])
        x = x[x["GNSS N"] >= 0]

        if not x.empty:
            pooled.extend(x["GNSS N"].astype(float).tolist())

    pooled = np.array(pooled, dtype=float)
    pooled = pooled[np.isfinite(pooled)]

    if pooled.size == 0:
        print("[GLOBAL] No valid GNSS N values found; GNSS N filtering disabled.")
        return None

    x = np.log1p(pooled)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    sigma = 1.4826 * mad

    if not np.isfinite(sigma) or sigma <= 0:
        threshold = float(np.nanmin(pooled))
    else:
        threshold = float(np.expm1(med - k * sigma))

    if abs_min is not None:
        threshold = max(threshold, float(abs_min))

    return threshold


def build_apc_lookup(offset_df: pd.DataFrame):
    offset_df = offset_df.copy()
    offset_df["station"] = offset_df["station"].astype(str).str.upper()
    offset_df["year"] = safe_numeric(offset_df["year"]).astype("Int64")
    offset_df["gps_l1_up_m"] = safe_numeric(offset_df["gps_l1_up_m"])

    station_unique = (
        offset_df.groupby("station")["gps_l1_up_m"]
        .apply(lambda s: sorted(set(float(v) for v in s.dropna())))
        .reset_index(name="unique_offsets")
    )

    station_unique["station_l1_offset_m"] = station_unique["unique_offsets"].apply(
        lambda x: x[0] if len(x) == 1 else np.nan
    )

    offset_lookup = offset_df[
        ["station", "year", "gps_l1_up_m", "offset_status"]
    ].copy()

    return station_unique, offset_lookup


def apply_l1_apc_correction(df_station, station_unique, offset_lookup):
    """
    Apply the station/year L1 APC correction.

    Fallback rule
    -------------
    If the requested year has no readable RINEX/ANTEX result, use the APC
    correction from the nearest year with a valid value for the same station.
    This avoids treating non-operational years or a single unreadable RINEX
    file as a fatal problem.

    When two valid years are equally close, the earlier year is selected.
    """
    df = df_station.copy()

    station_name = str(df["STN"].iloc[0]).upper()

    df["year"] = pd.to_datetime(
        df["YEAR-MM-DD"],
        errors="coerce",
    ).dt.year.astype("Int64")

    if "Adjusted_original(m)" not in df.columns:
        df["Adjusted_original(m)"] = df["Adjusted(m)"]

    year_offsets = offset_lookup[
        offset_lookup["station"] == station_name
    ].copy()

    year_offsets["year"] = safe_numeric(year_offsets["year"]).astype("Int64")
    year_offsets["gps_l1_up_m"] = safe_numeric(
        year_offsets["gps_l1_up_m"]
    )

    exact_offsets = year_offsets[
        ["year", "gps_l1_up_m", "offset_status"]
    ].copy()

    df = df.merge(
        exact_offsets,
        on="year",
        how="left",
    )

    df["APC_correction_m"] = df["gps_l1_up_m"]
    df["APC_reference_year"] = df["year"]
    df["APC_correction_source"] = np.where(
        df["gps_l1_up_m"].notna(),
        "year_specific",
        "missing",
    )

    valid_offsets = (
        year_offsets.dropna(subset=["year", "gps_l1_up_m"])
        .sort_values("year")
        .drop_duplicates(subset=["year"], keep="first")
    )

    if not valid_offsets.empty:
        valid_years = valid_offsets["year"].astype(int).to_numpy()
        valid_values = valid_offsets["gps_l1_up_m"].astype(float).to_numpy()

        missing_mask = df["APC_correction_m"].isna() & df["year"].notna()

        for idx in df.index[missing_mask]:
            target_year = int(df.at[idx, "year"])

            # np.argmin returns the first minimum, so sorting by year above
            # makes an equal-distance tie choose the earlier year.
            nearest_index = int(np.argmin(np.abs(valid_years - target_year)))
            reference_year = int(valid_years[nearest_index])
            reference_value = float(valid_values[nearest_index])

            df.at[idx, "APC_correction_m"] = reference_value
            df.at[idx, "APC_reference_year"] = reference_year
            df.at[idx, "APC_correction_source"] = "nearest_station_year"

    df["Adjusted(m)"] = (
        df["Adjusted_original(m)"] + df["APC_correction_m"]
    )

    return df.drop(
        columns=["gps_l1_up_m", "offset_status"],
        errors="ignore",
    )


# ============================================================
# UNCERTAINTY PROPAGATION
# ============================================================

def add_surface_elevation_uncertainty(df: pd.DataFrame) -> pd.DataFrame:
    """
    Propagate the daily GNSS and RH uncertainties into the derived surface
    elevation.

    Definitions used here
    ---------------------
    Original surface elevation:
        Adjusted_original(m) = GNSS mean daily height - RH_surface_height_m

    APC-corrected surface elevation:
        Adjusted(m) = Adjusted_original(m) + APC_correction_m

    Uncertainty propagation:
        sigma_surface = sqrt(sigma_GNSS^2 + sigma_RH^2)

    The APC correction is treated as a deterministic offset. No APC uncertainty
    term is included, so the original and corrected surface elevations have the
    same propagated uncertainty.

    Notes
    -----
    - "GNSS std" is used directly as the daily GNSS uncertainty.
    - "RH_std_filtered_m" is used directly as the daily RH uncertainty.
    - If either uncertainty is missing, the propagated uncertainty is left NaN.
    """
    output = df.copy()

    output["GNSS_uncertainty_m"] = safe_numeric(output["GNSS std"])
    output["RH_uncertainty_m"] = safe_numeric(
        output["RH_std_filtered_m"]
    )

    valid_uncertainty = (
        output["GNSS_uncertainty_m"].notna()
        & output["RH_uncertainty_m"].notna()
    )

    output["Adjusted_original_uncertainty_m"] = np.nan
    output.loc[
        valid_uncertainty,
        "Adjusted_original_uncertainty_m",
    ] = np.sqrt(
        output.loc[valid_uncertainty, "GNSS_uncertainty_m"] ** 2
        + output.loc[valid_uncertainty, "RH_uncertainty_m"] ** 2
    )

    # No uncertainty is assigned to the APC correction.
    output["Adjusted_uncertainty_m"] = (
        output["Adjusted_original_uncertainty_m"]
    )

    return output


# ============================================================
# PLOTTING
# ============================================================

def save_station_plot(df: pd.DataFrame, station: str, plot_dir: Path) -> Path:
    """
    Plot three daily quantities:

    Orange, left axis:
        Original GNSS elevation before subtracting RH and before applying APC.
        This is "GNSS mean daily height".

    Black, left axis:
        Final surface elevation after subtracting RH and applying the L1 APC
        correction. This is "Adjusted(m)".

    Red, right axis:
        Final classified RH used in the correction. This is
        "RH_surface_height_m".
    """
    plot_df = df.copy()
    plot_df["YEAR-MM-DD"] = pd.to_datetime(
        plot_df["YEAR-MM-DD"],
        errors="coerce",
    )
    plot_df = (
        plot_df.dropna(subset=["YEAR-MM-DD"])
        .sort_values("YEAR-MM-DD")
    )

    output = plot_dir / (
        f"{station.lower()}_raw_gnss_rh_and_final_corrected.png"
    )

    fig, ax_elev = plt.subplots(figsize=(15, 7))

    ax_elev.scatter(
        plot_df["YEAR-MM-DD"],
        plot_df["GNSS mean daily height"],
        s=10,
        alpha=0.75,
        color="orange",
        label="Original GNSS elevation",
    )

    ax_elev.scatter(
        plot_df["YEAR-MM-DD"],
        plot_df["Adjusted(m)"],
        s=10,
        alpha=0.75,
        color="black",
        label="Final elevation after RH and APC corrections",
    )

    ax_elev.set_xlabel("Date")
    ax_elev.set_ylabel("Elevation (m)")
    ax_elev.grid(True, alpha=0.3)

    ax_rh = ax_elev.twinx()

    ax_rh.scatter(
        plot_df["YEAR-MM-DD"],
        plot_df["RH_surface_height_m"],
        s=10,
        alpha=0.75,
        color="red",
        label="Final classified RH",
    )

    ax_rh.set_ylabel("Reflector height (m)", color="red")
    ax_rh.tick_params(axis="y", colors="red")
    ax_rh.spines["right"].set_color("red")
    ax_rh.yaxis.label.set_color("red")

    handles_1, labels_1 = ax_elev.get_legend_handles_labels()
    handles_2, labels_2 = ax_rh.get_legend_handles_labels()

    ax_elev.legend(
        handles_1 + handles_2,
        labels_1 + labels_2,
        loc="best",
    )

    ax_elev.set_title(
        f"{station.upper()}: raw GNSS elevation, classified RH, "
        "and final corrected elevation"
    )

    fig.tight_layout()
    fig.savefig(output, dpi=200, bbox_inches="tight")
    plt.close(fig)

    return output


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    print("========== CLASSIFIED RH + L1 APC + UNCERTAINTY ==========")
    print(f"RH source root: {RH_ROOT}")
    print(f"Regular GNSS daily averages: {GNSS_DAILY_POSITION_DIR}")
    print("RINEX search sources:")
    for source in RINEX_SOURCES:
        print(f"  - {source['name']}: {source['root']}")
    print(f"Requested stations: {len(STATIONS_ONLY)}")
    print(f"Output root: {OUT_DIR}")

    download_antex_if_needed(ATX_PATH, ANTEX_URLS)

    gnss_all, gnss_source_diag = load_all_gnss_daily(
        daily_position_dir=GNSS_DAILY_POSITION_DIR,
        stations=STATIONS_ONLY,
    )

    gnss_source_diag.to_csv(
        DIAG_DIR / "daily_position_average_load_diagnostics.csv",
        index=False,
    )

    requested_lower = {s.lower() for s in STATIONS_ONLY}
    gnss_all = gnss_all[gnss_all["STN"].isin(requested_lower)].copy()

    stations_in_gnss = sorted(
        gnss_all["STN"].dropna().str.upper().unique().tolist()
    )

    missing_from_gnss = sorted(
        set(s.upper() for s in STATIONS_ONLY) - set(stations_in_gnss)
    )

    print(f"[GNSS] Requested stations present: {len(stations_in_gnss)}")
    if missing_from_gnss:
        print(f"[GNSS] Missing requested stations: {missing_from_gnss}")

    rinex_df = build_rinex_antenna_table(stations_in_gnss)
    rinex_df.to_csv(
        DIAG_DIR / "rinex_station_year_antenna_and_bands.csv",
        index=False,
    )

    print("\n[RINEX] read_status counts:")
    print(rinex_df["read_status"].value_counts(dropna=False).to_string())

    rinex_summary = summarize_rinex_coverage(rinex_df)
    rinex_summary.to_csv(
        DIAG_DIR / "rinex_station_coverage_summary.csv",
        index=False,
    )

    rinex_missing = rinex_summary[
        rinex_summary["coverage_status"] != "complete"
    ].copy()

    rinex_missing.to_csv(
        DIAG_DIR / "stations_missing_rinex_files.csv",
        index=False,
    )

    print_rinex_coverage_report(rinex_summary)

    print("\n[RINEX] Coverage diagnostics written to:")
    print(f"        {DIAG_DIR / 'rinex_station_coverage_summary.csv'}")
    print(f"        {DIAG_DIR / 'stations_missing_rinex_files.csv'}")

    offset_df = build_l1_offset_table(rinex_df, ATX_PATH)
    offset_df.to_csv(
        DIAG_DIR / "station_year_L1_vertical_offsets_from_antex.csv",
        index=False,
    )

    print("\n[APC] offset_status counts:")
    print(offset_df["offset_status"].value_counts(dropna=False).to_string())

    station_unique, offset_lookup = build_apc_lookup(offset_df)

    rh_files = discover_rh_files_for_stations(RH_ROOT, STATIONS_ONLY)
    stations_with_rh = sorted(st.upper() for st in rh_files)
    missing_rh_requested = sorted(set(s.upper() for s in STATIONS_ONLY) - set(stations_with_rh))

    print(f"\n[RH] Requested stations with classified daily files: {len(rh_files)}")
    if missing_rh_requested:
        print(f"[RH] Requested stations missing classified daily files: {missing_rh_requested}")

    discovery_rows = []
    for station in STATIONS_ONLY:
        st_upper = station.upper()
        st_lower = station.lower()
        expected_path = (
            RH_ROOT / st_lower / f"{st_lower}_daily_summary_classified.csv"
        )
        discovery_rows.append({
            "station": st_upper,
            "requested": True,
            "present_in_gnss_daily": st_upper in stations_in_gnss,
            "classified_rh_file_found": st_lower in rh_files,
            "expected_rh_path": str(expected_path),
        })

    discovery_df = pd.DataFrame(discovery_rows)

    rh_all = load_all_rh(rh_files)

    rh_merge_cols = [
        "YEAR-MM-DD",
        "STN",
        "RH_surface_height_m",
        "RH median daily height",
        "daily_filtered_valid",
        "rh_source_file",
    ]

    for optional in [
        "RH_median_filtered_m",
        "RH_mean_filtered_m",
        "RH_std_filtered_m",
        "RH_N_filtered",
        "retained_fraction_of_all_power",
    ]:
        if optional in rh_all.columns:
            rh_merge_cols.append(optional)

    merged = gnss_all.merge(
        rh_all[rh_merge_cols],
        on=["YEAR-MM-DD", "STN"],
        how="inner",
    )

    merged = merged[merged["YEAR-MM-DD"] >= START_DATE].copy()

    if merged.empty:
        raise RuntimeError(
            "GNSS/classified-RH merge is empty. Check dates, station names, "
            "RH_ROOT, and the GNSS daily file."
        )

    # Surface elevation before APC correction:
    #     GNSS antenna elevation - final classified reflector height
    merged["Adjusted(m)"] = (
        merged["GNSS mean daily height"]
        - merged["RH_surface_height_m"]
    )

    station_dfs = {
        st: df.sort_values("YEAR-MM-DD").reset_index(drop=True)
        for st, df in merged.groupby("STN")
    }

    merged_station_set = set(st.upper() for st in station_dfs.keys())
    discovery_df["present_in_final_gnss_rh_merge"] = discovery_df["station"].isin(
        merged_station_set
    )
    discovery_df["missing_reason"] = discovery_df.apply(
        lambda row: (
            "missing_from_gnss_daily" if not row["present_in_gnss_daily"]
            else "missing_classified_rh" if not row["classified_rh_file_found"]
            else "no_date_overlap_after_inner_join" if not row["present_in_final_gnss_rh_merge"]
            else ""
        ),
        axis=1,
    )
    discovery_df.to_csv(
        DIAG_DIR / "requested_station_discovery.csv",
        index=False,
    )

    print(f"[LOAD] GNSS rows after station restriction: {len(gnss_all)}")
    print(f"[LOAD] Accepted classified RH rows: {len(rh_all)}")
    print(f"[MERGE] Matched GNSS/RH rows: {len(merged)}")
    print(f"[MERGE] Stations after inner join: {len(station_dfs)}")
    print("[MERGE] Requested-station diagnostics written to:")
    print(f"        {DIAG_DIR / 'requested_station_discovery.csv'}")

    print("[GLOBAL] GNSS N filtering disabled by configuration.")

    summary_rows = []

    for idx, (st, df0) in enumerate(sorted(station_dfs.items()), start=1):
        print(f"\n[{idx}/{len(station_dfs)}] Processing {st.upper()}")

        df = (
            df0.dropna(subset=["YEAR-MM-DD"])
            .sort_values("YEAR-MM-DD")
            .reset_index(drop=True)
        )

        if st in EXCLUDE_BY_STATION:
            before = len(df)

            for start_str, end_str in EXCLUDE_BY_STATION[st]:
                mask = df["YEAR-MM-DD"].between(
                    pd.to_datetime(start_str),
                    pd.to_datetime(end_str),
                    inclusive="both",
                )
                df = df.loc[~mask].copy()

            print(f"  [EXCLUDE] removed {before - len(df)} row(s)")

        for col in [
            "LATDD",
            "LONDD",
            "HGT(m)",
            "GNSS mean daily height",
            "GNSS std",
            "GNSS N",
            "RH_surface_height_m",
            "RH_median_filtered_m",
            "RH_mean_filtered_m",
            "RH_std_filtered_m",
            "RH_N_filtered",
            "retained_fraction_of_all_power",
            "Adjusted(m)",
        ]:
            if col in df.columns:
                df[col] = safe_numeric(df[col])

        df = df.dropna(
            subset=[
                "YEAR-MM-DD",
                "GNSS mean daily height",
                "RH_surface_height_m",
                "Adjusted(m)",
            ]
        ).copy()

        if df.empty:
            print("  [SKIP] no valid matched rows")
            summary_rows.append({
                "station": st.upper(),
                "status": "no_valid_rows",
            })
            continue

        # GNSS N is retained as metadata when available, but it is not used
        # to filter or reject any daily observations.
        removed_by_n = 0

        df = df.sort_values("YEAR-MM-DD").reset_index(drop=True)

        df["rz_roll"] = rolling_hampel(
            df["Adjusted(m)"],
            window=ROLL_WINDOW_DAYS,
            min_periods=MIN_PERIODS,
        )

        dY = df["Adjusted(m)"].diff()
        dT = df["YEAR-MM-DD"].diff().dt.days.clip(lower=1)
        df["rate_m_per_day"] = (dY / dT).abs()

        df["is_outlier_hampel"] = df["rz_roll"].abs() > MAD_THRESH
        df["is_outlier_rate"] = df["rate_m_per_day"] > RATE_THRESH_MPD
        df["is_outlier_local"] = (
            df["is_outlier_hampel"] | df["is_outlier_rate"]
        )

        df_apc = apply_l1_apc_correction(
            df,
            station_unique=station_unique,
            offset_lookup=offset_lookup,
        )

        df_apc = add_surface_elevation_uncertainty(df_apc)

        # Plot before date columns are converted to strings for CSV output.
        plot_path = save_station_plot(df_apc, st, PLOT_DIR)

        df_apc["YEAR-MM-DD"] = pd.to_datetime(
            df_apc["YEAR-MM-DD"],
            errors="coerce",
        ).dt.strftime("%Y-%m-%d")

        df_apc["date_only"] = pd.to_datetime(
            df_apc["date_only"],
            errors="coerce",
        ).dt.strftime("%Y-%m-%d")

        df_apc["STN"] = df_apc["STN"].astype(str).str.upper()
        df_apc = df_apc[[c for c in FINAL_COLS_OUT if c in df_apc.columns]]

        out_csv = APC_OUT_DIR / f"{st}_classified_rh_apc_corrected.csv"
        df_apc.to_csv(out_csv, index=False)

        print(f"  [WROTE] {out_csv}")
        print(f"  [PLOT]  {plot_path}")
        print(
            "  [UNCERTAINTY] "
            f"available={int(df_apc['Adjusted_uncertainty_m'].notna().sum())} / "
            f"{len(df_apc)}, "
            f"missing_GNSS_std={int(df_apc['GNSS_uncertainty_m'].isna().sum())}, "
            f"missing_RH_std={int(df_apc['RH_uncertainty_m'].isna().sum())}"
        )

        summary_rows.append({
            "station": st.upper(),
            "status": "ok",
            "n_rows": len(df_apc),
            "date_min": df_apc["YEAR-MM-DD"].min(),
            "date_max": df_apc["YEAR-MM-DD"].max(),
            "n_local_outliers": int(df_apc["is_outlier_local"].sum()),
            "n_missing_apc": int(df_apc["APC_correction_m"].isna().sum()),
            "n_missing_gnss_uncertainty": int(
                df_apc["GNSS_uncertainty_m"].isna().sum()
            ),
            "n_missing_rh_uncertainty": int(
                df_apc["RH_uncertainty_m"].isna().sum()
            ),
            "n_missing_final_uncertainty": int(
                df_apc["Adjusted_uncertainty_m"].isna().sum()
            ),
            "output_file": str(out_csv),
            "plot_file": str(plot_path),
        })

    summary_df = pd.DataFrame(summary_rows)
    if not summary_df.empty and "station" in summary_df.columns:
        summary_df = summary_df.sort_values("station").reset_index(drop=True)

    summary_path = DIAG_DIR / "final_processing_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print("\n========== COMPLETE ==========")
    print(f"Output root:          {OUT_DIR}")
    print(f"Corrected time series:{APC_OUT_DIR}")
    print(f"Plots:                {PLOT_DIR}")
    print(f"Diagnostics:          {DIAG_DIR}")
    print(f"Summary:              {summary_path}")

    missing_rinex_stations = (
        rinex_missing["station"].astype(str).tolist()
        if not rinex_missing.empty
        else []
    )

    print(
        f"Stations with incomplete RINEX coverage: "
        f"{len(missing_rinex_stations)}"
    )

    if missing_rinex_stations:
        print(", ".join(missing_rinex_stations))

    print("\nStatus counts:")
    if "status" in summary_df.columns:
        print(summary_df["status"].value_counts(dropna=False).to_string())


if __name__ == "__main__":
    main()


## Summarize daily reflector-height counts

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt

# NEW GNSS root
ROOT_GNSS = "/Volumes/cornice/marquardt/data/GNSS"

# Where your per-station daily RH stats CSVs live
OUT_DIR = os.path.join(ROOT_GNSS, "GNSSAdjusted")

# Pick up all per-station daily RH stats CSVs
pattern = os.path.join(OUT_DIR, "*_daily_rh_stats.csv")
files = sorted(glob.glob(pattern))
files = [
    fpath for fpath in files
    if os.path.basename(fpath).split("_")[0].upper() in STATIONS_ONLY
]

print(f"[LOAD] Found {len(files)} daily RH stats files in: {OUT_DIR}")
if len(files) == 0:
    print(f"[LOAD] Pattern used: {pattern}")

all_frames = []
stations_included = set()

for fpath in files:
    base = os.path.basename(fpath)
    st_tag = base.split("_")[0].upper()  # 'gz01_daily_rh_stats.csv' -> 'GZ01'
    print(f"  Loading {base} (station {st_tag})")

    df = pd.read_csv(fpath)

    # Basic sanity checks
    if "N_heights" not in df.columns:
        print(f"    ⚠️ Skipping {base}: no 'N_heights' column")
        continue

    # Parse Date (guard)
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Ensure numeric
    df["N_heights"] = pd.to_numeric(df["N_heights"], errors="coerce")

    # Keep valid rows
    df = df.dropna(subset=["N_heights"]).copy()
    if df.empty:
        print(f"    ⚠️ Skipping {base}: no valid numeric N_heights after cleaning")
        continue

    df["Station"] = st_tag
    stations_included.add(st_tag)
    all_frames.append(df[["Date", "Station", "N_heights"]])

if not all_frames:
    raise RuntimeError("No valid daily RH stats files with N_heights found.")

all_rh = pd.concat(all_frames, ignore_index=True)

# ---------------------------
# Totals / station list
# ---------------------------
total_reflector_heights = int(all_rh["N_heights"].sum())
stations_sorted = sorted(stations_included)

print("\n========== SUMMARY ==========")
print(f"Stations included (n={len(stations_sorted)}): {', '.join(stations_sorted)}")
print(f"Total daily entries loaded: {len(all_rh)}")
print(f"Total reflector heights processed (sum of N_heights): {total_reflector_heights}")
print(f"N_heights min={all_rh['N_heights'].min()}, max={all_rh['N_heights'].max()}, median={all_rh['N_heights'].median()}")

# ---------------------------
# Histogram of N_heights
# ---------------------------
max_N = int(all_rh["N_heights"].max())
bin_width = 5
bins = range(0, max_N + bin_width, bin_width)

plt.figure(figsize=(10, 6))
plt.hist(all_rh["N_heights"], bins=bins, edgecolor="black", alpha=0.8)
plt.xlabel("Number of reflector heights per day (N_heights)")
plt.ylabel("Count of days")
plt.title("Histogram of daily reflector height counts across all stations")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

out_png = os.path.join(OUT_DIR, "hist_N_heights_all_stations.png")
plt.savefig(out_png, dpi=200)
plt.close()
print(f"\n[EXPORT] Wrote histogram → {out_png}")